In [32]:
import os
from llama_index.core import Settings
from llama_index.llms.groq import Groq

In [33]:
Settings.llm = Groq(model="llama-3.3-70b-versatile",
                    api_key=os.environ.get("GROQ_API_KEY"))
import pandas as pd

df_vendas = pd.read_csv("vendas.csv")

In [34]:
from llama_index.core import PromptTemplate
from llama_index.experimental.query_engine.pandas import PandasInstructionParser

In [35]:
instruction_str = (
    "1. Converta a consulta para código Python executável usando Pandas.\n"
    "2. A linha final do código deve ser uma expressão Python que possa ser chamada com a função `eval()`.\n"
    "3. O código deve representar uma solução para a consulta.\n"
    "4. IMPRIMA APENAS A EXPRESSÃO.\n"
    "5. Não coloque a expressão entre aspas.\n")

# Prompt que será enviado ao modelo para que ela gere o código Pandas desejado
pandas_prompt_str = (
    "Você está trabalhando com um dataframe do pandas em Python chamado `df`.\n"
    "{colunas_detalhes}\n\n"
    "Este é o resultado de `print(df.head())`:\n"
    "{df_str}\n\n"
    "Siga estas instruções:\n"
    "{instruction_str}\n"
    "Consulta: {query_str}\n\n"
    "Expressão:"
)

# Prompt para guiar o modelo a sintetizar uma resposta com base nos resultados obtidos pela consulta Pandas
response_synthesis_prompt_str = (
   "Dada uma pergunta de entrada, atue como analista de dados e elabore uma resposta a partir dos resultados da consulta.\n"
   "Responda de forma natural, sem introduções como 'A resposta é:' ou algo semelhante.\n"
   "Consulta: {query_str}\n\n"
   "Instruções do Pandas (opcional):\n{pandas_instructions}\n\n"
   "Saída do Pandas: {pandas_output}\n\n"
   "Resposta:"
   "Ao final, exibir o código usado para gerar a resposta, no formato: O código utilizado foi {pandas_instructions}"
)

In [36]:
def descreve_colunas(df):
    descricao = "\n".join([f"`{col}`: {str(df[col].dtype)}" for col in df.columns])
    return "Detalhes das colunas do Dataframe:\n", descricao

In [37]:
pandas_prompt = PromptTemplate(pandas_prompt_str).partial_format(
    instruction_str=instruction_str,
    colunas_detalhes=descreve_colunas(df_vendas), 
                     df_str=df_vendas.head(5)
)
pandas_output_parser = PandasInstructionParser(df_vendas)

response_synthesis_prompt = PromptTemplate(response_synthesis_prompt_str)

llm = Groq(model="llama-3.3-70b-versatile", api_key=os.environ.get("GROQ_API_KEY"))

In [38]:
from llama_index.core.query_pipeline import (QueryPipeline as QP, Link, InputComponent)

In [39]:
qp = QP(
    modules = {
        "input":InputComponent(),
        "pandas_prompt": pandas_prompt,
        "llm1": llm,
        "pandas_output_parser": pandas_output_parser,
        "response_synthesis_prompt": response_synthesis_prompt,
        "llm2": llm
    },
    verbose=True
)

In [40]:
qp.add_chain(["input", "pandas_prompt", "llm1", "pandas_output_parser"])
qp.add_links(
    [
        Link("input", "response_synthesis_prompt", dest_key="query_str"),
        Link("llm1", "response_synthesis_prompt", dest_key="pandas_instructions"),
        Link("pandas_output_parser", "response_synthesis_prompt", dest_key="pandas_output")
    ]
)
qp.add_link("response_synthesis_prompt", "llm2")

### Consultas no Pipeline

In [41]:
response = qp.run(query_str="Qual é o vendedor com maior volume de vendas?")

> Running module input with input: 
query_str: Qual é o vendedor com maior volume de vendas?

> Running module pandas_prompt with input: 
query_str: Qual é o vendedor com maior volume de vendas?

> Running module llm1 with input: 
messages: Você está trabalhando com um dataframe do pandas em Python chamado `df`.
('Detalhes das colunas do Dataframe:\n', '`ID_compra`: object\n`filial`: object\n`cidade`: object\n`tipo_cliente`: object\n`gen...

> Running module pandas_output_parser with input: 
input: assistant: df.groupby('filial')['total'].sum().idxmax()

> Running module response_synthesis_prompt with input: 
query_str: Qual é o vendedor com maior volume de vendas?
pandas_instructions: assistant: df.groupby('filial')['total'].sum().idxmax()
pandas_output: C

> Running module llm2 with input: 
messages: Dada uma pergunta de entrada, atue como analista de dados e elabore uma resposta a partir dos resultados da consulta.
Responda de forma natural, sem introduções como 'A resposta é:' ou a

In [42]:
import textwrap

texto = response.message.content
texto_formatado = textwrap.fill(texto, width=100)
print(texto_formatado)

O vendedor com maior volume de vendas é a filial C. Isso indica que, dentre as filiais analisadas, a
filial C apresentou o maior total de vendas.  O código utilizado foi
df.groupby('filial')['total'].sum().idxmax()


In [43]:
qp.run(query_str="Quais fatores podem explicar o alto desempenho desse vendedor?")

> Running module input with input: 
query_str: Quais fatores podem explicar o alto desempenho desse vendedor?

> Running module pandas_prompt with input: 
query_str: Quais fatores podem explicar o alto desempenho desse vendedor?

> Running module llm1 with input: 
messages: Você está trabalhando com um dataframe do pandas em Python chamado `df`.
('Detalhes das colunas do Dataframe:\n', '`ID_compra`: object\n`filial`: object\n`cidade`: object\n`tipo_cliente`: object\n`gen...

> Running module pandas_output_parser with input: 
input: assistant: df[['filial', 'cidade', 'tipo_cliente', 'genero', 'tipo_produto', 'forma_pagamento']].value_counts()

> Running module response_synthesis_prompt with input: 
query_str: Quais fatores podem explicar o alto desempenho desse vendedor?
pandas_instructions: assistant: df[['filial', 'cidade', 'tipo_cliente', 'genero', 'tipo_produto', 'forma_pagamento']].value_counts()
pandas_output: filial  cidade                 tipo_cliente  genero     tipo_produto   

ChatResponse(message=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, content='Para entender os fatores que contribuem para o alto desempenho de um vendedor, é importante analisar os dados disponíveis sobre as vendas, incluindo a filial, cidade, tipo de cliente, gênero do cliente, tipo de produto vendido e forma de pagamento utilizada. \n\nA partir dos dados fornecidos, podemos observar que:\n\n- A filial "C" em São Caetano apresenta um alto desempenho, com 13 vendas realizadas com clientes normais do gênero feminino que compraram eletrônicos e pagaram em dinheiro.\n- A filial "B" em São Bernardo do Campo também tem um desempenho notável, com 11 vendas para clientes normais do gênero masculino que compraram produtos para casa e pagaram com carteira digital.\n- Além disso, a filial "A" em Santo André tem 10 vendas para clientes normais do gênero feminino que compraram eletrônicos e pagaram com cartão de crédito, assim como 10 vendas para membros do gênero feminino que compraram al

In [45]:
# Pipeline de consulta
'''Função para obter uma descrição das colunas do DataFrame'''
def descricao_colunas(df):
    descricao = '\n'.join([f"`{col}`: {str(df[col].dtype)}" for col in df.columns])
    return "Aqui estão os detalhes das colunas do dataframe:\n" + descricao

'''Definição de módulos da pipeline'''
def pipeline_consulta(df):
    instruction_str = (
        "1. Converta a consulta para código Python executável usando Pandas.\n"
        "2. A linha final do código deve ser uma expressão Python que possa ser chamada com a função `eval()`.\n"
        "3. O código deve representar uma solução para a consulta.\n"
        "4. IMPRIMA APENAS A EXPRESSÃO.\n"
        "5. Não coloque a expressão entre aspas.\n")

    pandas_prompt_str = (
        "Você está trabalhando com um dataframe do pandas em Python chamado `df`.\n"
        "{colunas_detalhes}\n\n"
        "Este é o resultado de `print(df.head())`:\n"
        "{df_str}\n\n"
        "Siga estas instruções:\n"
        "{instruction_str}\n"
        "Consulta: {query_str}\n\n"
        "Expressão:"
)

    response_synthesis_prompt_str = (
       "Dada uma pergunta de entrada, atue como analista de dados e elabore uma resposta a partir dos resultados da consulta.\n"
       "Responda de forma natural, sem introduções como 'A resposta é:' ou algo semelhante.\n"
       "Consulta: {query_str}\n\n"
       "Instruções do Pandas (opcional):\n{pandas_instructions}\n\n"
       "Saída do Pandas: {pandas_output}\n\n"
       "Resposta: \n\n"
       "Ao final, exibir o código usado em para gerar a resposta, no formato: O código utilizado foi `{pandas_instructions}`"
    )

    pandas_prompt = PromptTemplate(pandas_prompt_str).partial_format(
    instruction_str=instruction_str,
    df_str=df.head(5),
    colunas_detalhes=descricao_colunas(df)
)

    pandas_output_parser = PandasInstructionParser(df)
    response_synthesis_prompt = PromptTemplate(response_synthesis_prompt_str)

    '''Criação do Query Pipeline'''
    qp = QP(
        modules={
            "input": InputComponent(),
            "pandas_prompt": pandas_prompt,
            "llm1": llm,
            "pandas_output_parser": pandas_output_parser,
            "response_synthesis_prompt": response_synthesis_prompt,
            "llm2": llm,
        },
        verbose=True,
    )
    qp.add_chain(["input", "pandas_prompt", "llm1", "pandas_output_parser"])
    qp.add_links(
        [
            Link("input", "response_synthesis_prompt", dest_key="query_str"),
            Link("llm1", "response_synthesis_prompt", dest_key="pandas_instructions"),
            Link("pandas_output_parser", "response_synthesis_prompt", dest_key="pandas_output"),
        ]
    )
    qp.add_link("response_synthesis_prompt", "llm2")
    return qp

### Interface com o Gradio

In [46]:
import gradio as gr

def load_data(file_path, df_state):
    if file_path is None or file_path == "":
        return "Faça o upload de um dataset para começar a análise", df_state
    try:
        df = pd.read_csv(file_path)
        return "Arquivo carregado com sucesso!", df
    except Exception as e:
        return f"Erro ao carregar o dataset: {str(e)}", df_state
    
def proccess_question(question, df_state):
    if df_state is not None and question:
        qp = pipeline_consulta(df_state)
        response = qp.run(query_str=question)
        return response.message.content
    return ""

with gr.Blocks() as app:
    in_file = gr.File(file_count="single",
                      type="filepath",
                      label="Upload CSV")
    status_upload = gr.Textbox(label="Status do Upload")
    in_question = gr.Textbox(label="Digite sua pergunta sobre os dados")
    bt_submit = gr.Button("Enviar")
    out_answer = gr.Textbox(label="Resposta")
    df_state = gr.State(value=None)
    
    in_file.change(fn=load_data,
                   inputs=[in_file, df_state],
                   outputs=[status_upload, df_state])
    
    bt_submit.click(fn=proccess_question,
                    inputs=[in_question, df_state],
                    outputs=out_answer)
app.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


> Running module input with input: 
query_str: qual o produto menos vendido

> Running module pandas_prompt with input: 
query_str: qual o produto menos vendido

> Running module llm1 with input: 
messages: Você está trabalhando com um dataframe do pandas em Python chamado `df`.
Aqui estão os detalhes das colunas do dataframe:
`ID_compra`: object
`filial`: object
`cidade`: object
`tipo_cliente`: object
`...

> Running module pandas_output_parser with input: 
input: assistant: df.loc[df['quantidade'].idxmin()]['tipo_produto']

> Running module response_synthesis_prompt with input: 
query_str: qual o produto menos vendido
pandas_instructions: assistant: df.loc[df['quantidade'].idxmin()]['tipo_produto']
pandas_output: Moda

> Running module llm2 with input: 
messages: Dada uma pergunta de entrada, atue como analista de dados e elabore uma resposta a partir dos resultados da consulta.
Responda de forma natural, sem introduções como 'A resposta é:' ou algo semelhante...

Keyboard interrupti